<a href="https://colab.research.google.com/github/durga0000/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

if 'COLAB_GPU' in os.environ:
  print('[info] Running in google colab , installing requirements.')
  !pip install pyMuPDF
  !pip install tqdm
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers

!pip install torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

!pip install -U transformers sentence-transformers

In [4]:
import os

In [ ]:
import requests

pdf_path = 'human nutrition.pdf'

if not os.path.exists(pdf_path):
   print('File does not exist , downloaging...')

   url = "https://www.cur.ac.rw/mis/main/library/documents/book_file/digital-67fe1efc6feac4.39858496.pdf"

   filename = pdf_path

   response = requests.get(url)

   if response.status_code == 200:
       with open(filename , "wb") as file:
             file.write(response.content)
       print(f"file has been downloaded and saved as {filename}")
   else:
       print(f"failed to download the file . staus code: {response.status_code}")

else:
  print(f"file {pdf_path} exists.")


In [ ]:
import fitz
from tqdm.auto import tqdm

def text_formatter(text: str)->str:
    cleaned_text = text.replace("\n"," ").strip()

    return cleaned_text

def open_and_read_pdf(pdf_path:str)->list[dict]:

     doc  = fitz.open(pdf_path)
     pages_and_texts = []

     for page_number , page in tqdm(enumerate(doc)):
         text = page.get_text()
         text = text_formatter(text)

         pages_and_texts.append({"page_number":page_number-8,
                                 "physical_page": page_number,
                                 "page_char_count":len(text),
                                 "page_word_count":len(text.split(' ')),
                                 "page_sentence_count_raw":len(text.split(".  ")),
                                 "page_token_count":len(text)/4 ,
                                 "text": text
                                 })
     return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)

pages_and_texts[:10]


In [ ]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

In [ ]:
df.describe().round(2)

**Testing All 5 chunking stratigies ** :Fixed size chunking


In [ ]:
def chunk_text(text: str , chunk_size: int =500)->list:
    chunks = []
    current_chunk = ''
    words = text.split()

    for word in words:
      if len(current_chunk)+len(word)+1<=chunk_size:
         current_chunk+=(word+' ')
      else:
        chunks.append(current_chunk.strip())
        current_chunk=word + ' '

    if current_chunk:
       chunks.append(current_chunk.strip())

    return chunks


def chunk_pdf_pages(pages_and_texts:list,chunk_size: int =500)->list[dict]:
    all_chunks = []

    for page in pages_and_texts:
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = chunk_text(page_text,chunk_size)

        for i , chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_id":i,
                "chunk_char_count":len(chunk),
                "chunk_word_count":len(chunk.split()),
                 "chunk_roken_count":len(chunk)/4 ,
                "chunk_text":chunk

            })

    return all_chunks


chunked_pages = chunk_pdf_pages(pages_and_texts,chunk_size=500)

print(f"total chunks:{len(chunked_pages)}")





Chunks visualization

In [ ]:
for chunk in chunked_pages:
    print("+" + "-" * 98 + "+")
    print(f"| Page Number : {chunk['page_number']:<82}|")
    print(f"| Chunk ID    : {chunk['chunk_id']:<82}|")
    print(f"| Words       : {chunk['chunk_word_count']:<82}|")
    print(f"| Characters  : {chunk['chunk_char_count']:<82}|")
    print("+" + "-" * 98 + "+")

    import textwrap

    wrapped_text = textwrap.wrap(chunk['chunk_text'], width=96)

    for line in wrapped_text:
        print(f"| {line:<96}|")

    print("+" + "-" * 98 + "+")
    print()

## Semantic Chunking - Short Notes

### What is Semantic Chunking?

Semantic chunking splits text based on **meaning** rather than fixed character or token limits.

---

### How it Works

1. **Split text into sentences**

   ```python
   nltk.sent_tokenize(text)
   ```

2. **Convert each sentence into embeddings**

   ```python
   model.encode(sentences)
   ```

3. **Calculate similarity**

   ```python
   cosine_similarity(embedding1, embedding2)
   ```

4. **Group similar sentences**

   * Similarity ≥ threshold (e.g., 0.8) → Same chunk
   * Similarity < threshold → New chunk

5. **Check max token limit**

   * Prevent chunks from becoming too large.

---

### Flow

```text
Text
 ↓
Sentence Splitting
 ↓
Sentence Embeddings
 ↓
Cosine Similarity
 ↓
Group Similar Sentences
 ↓
Semantic Chunks
```

---

### Why Use It?

**Fixed Chunking**

```text
Chunk 1:
Python is a programming language.
Django is a web

Chunk 2:
framework used for web development.
```

Context is broken.

**Semantic Chunking**

```text
Chunk 1:
Python is a programming language.
Django is a web framework.

Chunk 2:
The heart pumps blood.
Blood circulates through the body.
```

Context is preserved.

---

### Components Used

* **NLTK** → Sentence splitting
* **SentenceTransformer** → Generate embeddings
* **Cosine Similarity** → Measure sentence similarity
* **Threshold (e.g., 0.8)** → Decide whether to continue or start a new chunk

---

### Advantages

✅ Preserves context and topic boundaries
✅ Better retrieval quality in RAG
✅ Groups related information together

### Disadvantages

❌ Slower than recursive chunking
❌ Requires embedding generation during chunking
❌ Higher computational cost

---

### Interview One-Liner

> Semantic chunking creates chunks by grouping semantically similar sentences using embeddings and cosine similarity, ensuring that each chunk contains information from the same topic rather than being split purely by size.


In [ ]:
!pip install -q --upgrade "sentence-transformers==3.0.1" "transformers>=4.41,<5" scikit-learn nltk

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk

nltk.download("punkt", quiet=True)

# Load model once
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")


def semantic_chunk_text(
    text: str,
    similarity_threshold: float = 0.8,
    max_tokens: int = 500
) -> list:
    """
    Split text into semantic chunks based on sentence similarity.
    """
    # breake the text into sentences we use nltk for that
    sentences = nltk.sent_tokenize(text)

    if not sentences:
        return []

    embeddings = semantic_model.encode(sentences)

    chunks = []
    current_chunk = [sentences[0]]
    current_embedding = embeddings[0]

    for i in range(1, len(sentences)):

        sim = cosine_similarity(
            [current_embedding],
            [embeddings[i]]
        )[0][0]

        chunk_token_count = len(" ".join(current_chunk)) // 4

        if sim >= similarity_threshold and chunk_token_count < max_tokens:
            current_chunk.append(sentences[i])

            current_embedding = np.mean(
                [current_embedding, embeddings[i]],
                axis=0
            )
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
            current_embedding = embeddings[i]

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks



    from tqdm.auto import tqdm


def semantic_chunk_pdf_pages(
    pages_and_texts: list,
    similarity_threshold: float = 0.8,
    max_tokens: int = 500
) -> list[dict]:
    """
    Takes PDF pages and splits them into semantic chunks.
    """

    all_chunks = []

    for page in tqdm(
        pages_and_texts,
        desc="Semantic chunking pages"
    ):

        page_number = page["page_number"]
        page_text = page["text"]

        chunks = semantic_chunk_text(
            page_text,
            similarity_threshold=similarity_threshold,
            max_tokens=max_tokens
        )

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough estimate
                "chunk_text": chunk
            })

    return all_chunks

In [ ]:
import nltk
nltk.download('punkt_tab')
semantic_chunks = semantic_chunk_pdf_pages(
    pages_and_texts,
    similarity_threshold=0.8,
    max_tokens=500
)

print(f"Total Semantic Chunks: {len(semantic_chunks)}")

The examples above demonstrate how to implement chunking strategies from scratch without using LangChain. Building these implementations manually helps us understand the underlying logic behind each chunking technique and allows us to customize the behavior based on our specific use case. Similarly, we can write our own implementations for other chunking strategies, such as Fixed Chunking, Recursive Chunking, Structural Chunking, and Structural + Recursive Chunking, giving us full control over how documents are split and processed for RAG applications.